In [ ]:
# Install required packages for enhanced model
# Run this cell first before training

import subprocess
import sys

packages = ['lpips', 'scipy']

for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

print("\n🎉 All required packages are ready!")

In [ ]:
# Option 1: Download dataset using kagglehub (requires Kaggle API setup)
# Uncomment if you want to use kagglehub:
# import kagglehub
# path = kagglehub.dataset_download("abdulhasibuddin/uc-merced-land-use-dataset")
# print("Path to dataset files:", path)

# Option 2: Manual download
# 1. Download from: https://www.kaggle.com/datasets/abdulhasibuddin/uc-merced-land-use-dataset
# 2. Extract to: C:\Users\MAROUAN\Downloads\UCMerced_LandUse
# 3. Or update dataset_path in the next cell to your extraction location

print("⚠️ Make sure the UC Merced dataset is downloaded and extracted locally!")
print("Dataset URL: https://www.kaggle.com/datasets/abdulhasibuddin/uc-merced-land-use-dataset")
print("Recommended path: C:\\Users\\MAROUAN\\Downloads\\UCMerced_LandUse")

In [ ]:
import os
import numpy as np
import math
import itertools
import datetime
import time
import cv2
import matplotlib.pyplot as plt
import json

import torchvision.transforms as transforms
from torchvision.utils import save_image
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torch.autograd import Variable

from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
from torchvision import models

# --- 1. CONFIGURATION ---
PROJECT_NAME = "Satellite_Context_Aware_Modern"
IMG_SIZE = 256
BATCH_SIZE = 16
LR = 0.0002
EPOCHS = 100
SAMPLE_INTERVAL = 200
CHECKPOINT_INTERVAL = 10
VAL_SPLIT = 0.1

# Configuration GPU
cuda = True if torch.cuda.is_available() else False
device = torch.device("cuda" if cuda else "cpu")
print(f"🖥️ Environnement : {device}")
if cuda:
    print(f"🎮 GPU : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"📊 Batch Size : {BATCH_SIZE}")

# Dossiers de sortie
output_dir = os.path.join(os.getcwd(), "output")
os.makedirs(os.path.join(output_dir, "images_modern"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "saved_models"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "metrics"), exist_ok=True)
print(f"📁 Output directory: {output_dir}")

# --- 2. DATASET & MASQUE NUAGEUX ---

class SatelliteDataset(Dataset):
    def __init__(self, root_dir, transforms_=None):
        self.transform = transforms_
        self.files = []
        for dirpath, _, filenames in os.walk(root_dir):
            for f in filenames:
                if f.lower().endswith(('.jpg', '.png', '.tif', '.jpeg')):
                    self.files.append(os.path.join(dirpath, f))
        print(f"✅ Images chargées : {len(self.files)}")
        if len(self.files) == 0:
            raise ValueError(f"❌ No images found in {root_dir}")

    def __getitem__(self, index):
        try:
            img = Image.open(self.files[index % len(self.files)]).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img
        except Exception as e:
            print(f"⚠️ Error: {e}")
            return torch.zeros((3, IMG_SIZE, IMG_SIZE))

    def __len__(self):
        return len(self.files)

def apply_cloud_mask(imgs):
    masked_imgs = imgs.clone()
    batch_size, _, height, width = imgs.shape
    
    for i in range(batch_size):
        mask = np.zeros((height, width), np.float32)
        
        num_strokes = np.random.randint(2, 5)
        for _ in range(num_strokes):
            x1, x2 = np.random.randint(0, width), np.random.randint(0, width)
            y1, y2 = np.random.randint(0, height), np.random.randint(0, height)
            thickness = np.random.randint(3, 12)
            cv2.line(mask, (x1, y1), (x2, y2), (1), thickness)
            
        num_circles = np.random.randint(1, 4)
        for _ in range(num_circles):
            x = np.random.randint(0, width)
            y = np.random.randint(0, height)
            radius = np.random.randint(3, 10)
            cv2.circle(mask, (x, y), radius, (1), -1)
        
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
        mask = (mask > 0.3).astype(np.float32)
        mask_tensor = torch.from_numpy(mask).to(imgs.device)
        
        masked_imgs[i, 0, mask_tensor == 1] = 1.0
        masked_imgs[i, 1, mask_tensor == 1] = 1.0
        masked_imgs[i, 2, mask_tensor == 1] = 1.0
        
    return masked_imgs

# Chargement Données
transforms_ = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset_path = r"C:\Users\MAROUAN\Downloads\Project\UCMerced_LandUse\Images"

alternative_paths = [
    r"C:\Users\MAROUAN\Downloads\Project\UCMerced_LandUse",
    r"C:\Users\MAROUAN\Downloads\UCMerced_LandUse\Images",
    r"C:\Users\MAROUAN\Downloads\UCMerced_LandUse",
]

if not os.path.exists(dataset_path):
    print(f"⚠️ Path not found: {dataset_path}")
    for alt_path in alternative_paths:
        if os.path.exists(alt_path):
            dataset_path = alt_path
            print(f"✅ Found dataset at: {dataset_path}")
            break
    else:
        raise FileNotFoundError("❌ Dataset not found!")

# Train/Val split
full_dataset = SatelliteDataset(dataset_path, transforms_=transforms_)
val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

print(f"📊 Train: {train_size}, Validation: {val_size}")

dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=cuda)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=cuda)

# --- 3. ARCHITECTURE ---

class ResidualBlock(nn.Module):
    def __init__(self, features, dilation=1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(dilation),
            nn.Conv2d(features, features, 3, padding=0, dilation=dilation),
            nn.InstanceNorm2d(features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3, padding=0, dilation=1),
            nn.InstanceNorm2d(features)
        )
    def forward(self, x):
        return x + self.block(x)

class ModernGenerator(nn.Module):
    def __init__(self, channels=3):
        super(ModernGenerator, self).__init__()
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, 64, 7), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.InstanceNorm2d(256), nn.ReLU(True)
        ]
        model += [ResidualBlock(256, dilation=2)]
        model += [ResidualBlock(256, dilation=4)]
        model += [ResidualBlock(256, dilation=8)]
        model += [ResidualBlock(256, dilation=1)]
        model += [
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, channels, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, channels=3):
        super(Discriminator, self).__init__()
        def discriminator_block(in_filters, out_filters, stride, normalize):
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride, 1)]
            if normalize: layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        layers = []
        layers.extend(discriminator_block(channels, 64, 2, False))
        layers.extend(discriminator_block(64, 128, 2, True))
        layers.extend(discriminator_block(128, 256, 2, True))
        layers.extend(discriminator_block(256, 512, 1, True))
        layers.append(nn.Conv2d(512, 1, 4, 1, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, img):
        return self.model(img)

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = models.vgg19(pretrained=True).features
        self.slice1 = nn.Sequential(*list(vgg[:5])).eval()
        self.slice2 = nn.Sequential(*list(vgg[5:10])).eval()
        self.slice3 = nn.Sequential(*list(vgg[10:19])).eval()
        for param in self.parameters():
            param.requires_grad = False
    
    def forward(self, x, y):
        x1, y1 = self.slice1(x), self.slice1(y)
        x2, y2 = self.slice2(x1), self.slice2(y1)
        x3, y3 = self.slice3(x2), self.slice3(y2)
        return F.l1_loss(x1, y1) + F.l1_loss(x2, y2) + F.l1_loss(x3, y3)

def weights_init_normal(m):
    if isinstance(m, nn.Conv2d): 
        torch.nn.init.normal_(m.weight.data, 0.0, 0.02)

generator = ModernGenerator().to(device)
discriminator = Discriminator().to(device)

print(f"🧠 Generator params: {sum(p.numel() for p in generator.parameters()):,}")
print(f"🧠 Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

generator.apply(weights_init_normal)
discriminator.apply(weights_init_normal)

adversarial_loss = torch.nn.MSELoss().to(device)
pixelwise_loss = torch.nn.L1Loss().to(device)
perceptual_loss = VGGPerceptualLoss().to(device)
lpips_loss = lpips.LPIPS(net='alex').to(device)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))

# --- 4. TRAINING ---

def save_sample(batches_done):
    imgs = next(iter(dataloader))
    imgs = imgs.to(device)
    masked_imgs = apply_cloud_mask(imgs)
    generator.eval()
    with torch.no_grad():
        gen_imgs = generator(masked_imgs)
    generator.train()
    sample = torch.cat((masked_imgs[:5], gen_imgs[:5], imgs[:5]), -2)
    save_image(sample, os.path.join(output_dir, "images_modern", f"{batches_done}.png"), nrow=5, normalize=True)

print("\n" + "="*60)
print("🚀 Starting training...")
print("="*60)

history = {
    'train_g_loss': [],
    'train_d_loss': [],
    'val_psnr': [],
    'val_ssim': [],
    'epochs': []
}

prev_time = time.time()
best_val_psnr = 0

for epoch in range(EPOCHS):
    for i, imgs in enumerate(dataloader):
        imgs = imgs.to(device)
        masked_imgs = apply_cloud_mask(imgs)

        # Train Generator
        optimizer_G.zero_grad()
        gen_imgs = generator(masked_imgs)
        
        pred_fake = discriminator(gen_imgs)
        valid = torch.ones(pred_fake.size(), device=device, requires_grad=False)
        fake = torch.zeros(pred_fake.size(), device=device, requires_grad=False)

        g_adv = adversarial_loss(pred_fake, valid)
        g_pixel = pixelwise_loss(gen_imgs, imgs)
        g_perceptual = perceptual_loss(gen_imgs, imgs)
        g_loss = 0.005 * g_adv + 0.5 * g_pixel + 0.495 * g_perceptual 

        g_loss.backward()
        optimizer_G.step()

        # Train Discriminator
        optimizer_D.zero_grad()
        real_loss = adversarial_loss(discriminator(imgs), valid)
        fake_loss = adversarial_loss(discriminator(gen_imgs.detach()), fake)
        d_loss = 0.5 * (real_loss + fake_loss)
        d_loss.backward()
        optimizer_D.step()

        # Log
        batches_done = epoch * len(dataloader) + i
        batches_left = EPOCHS * len(dataloader) - batches_done
        time_left = datetime.timedelta(seconds=batches_left * (time.time() - prev_time))
        prev_time = time.time()

        if i % 50 == 0:
            print(f"[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(dataloader)}] [D: {d_loss.item():.4f}] [G: {g_loss.item():.4f}] [ETA: {time_left}]")
        
        if batches_done % SAMPLE_INTERVAL == 0:
            save_sample(batches_done)
    
    # Validation
    history['train_g_loss'].append(g_loss.item())
    history['train_d_loss'].append(d_loss.item())
    history['epochs'].append(epoch)
    
    generator.eval()
    val_psnr_scores, val_ssim_scores = [], []
    with torch.no_grad():
        for val_imgs in val_dataloader:
            val_imgs = val_imgs.to(device)
            val_masked = apply_cloud_mask(val_imgs)
            val_gen = generator(val_masked)
            
            real_np = ((val_imgs + 1) / 2.0).cpu().numpy()
            gen_np = ((val_gen + 1) / 2.0).cpu().numpy()
            
            for j in range(real_np.shape[0]):
                r = np.transpose(real_np[j], (1, 2, 0))
                g = np.transpose(gen_np[j], (1, 2, 0))
                val_psnr_scores.append(psnr(r, g, data_range=1.0))
                val_ssim_scores.append(ssim(r, g, data_range=1.0, channel_axis=2))
    
    avg_val_psnr = np.mean(val_psnr_scores)
    avg_val_ssim = np.mean(val_ssim_scores)
    history['val_psnr'].append(avg_val_psnr)
    history['val_ssim'].append(avg_val_ssim)
    
    print(f"\n📊 Epoch {epoch} - Val PSNR: {avg_val_psnr:.2f}dB | Val SSIM: {avg_val_ssim:.4f}")
    
    if avg_val_psnr > best_val_psnr:
        best_val_psnr = avg_val_psnr
        best_model_path = os.path.join(output_dir, "saved_models", "generator_best.pth")
        torch.save(generator.state_dict(), best_model_path)
        print(f"✅ New best model! PSNR: {best_val_psnr:.2f}dB")
    
    if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
        checkpoint = {
            'epoch': epoch,
            'generator': generator.state_dict(),
            'discriminator': discriminator.state_dict(),
            'optimizer_G': optimizer_G.state_dict(),
            'optimizer_D': optimizer_D.state_dict(),
            'best_val_psnr': best_val_psnr
        }
        checkpoint_path = os.path.join(output_dir, "checkpoints", f"checkpoint_epoch_{epoch+1}.pth")
        torch.save(checkpoint, checkpoint_path)
        print(f"💾 Checkpoint saved at epoch {epoch+1}")
    
    generator.train()

# Save final model
model_save_path = os.path.join(output_dir, "saved_models", "generator_modern_final.pth")
torch.save(generator.state_dict(), model_save_path)
print(f"\n✅ Training complete! Saved to: {model_save_path}")

# Save history (convert numpy types to Python types)
history_json = {
    'train_g_loss': [float(x) for x in history['train_g_loss']],
    'train_d_loss': [float(x) for x in history['train_d_loss']],
    'val_psnr': [float(x) for x in history['val_psnr']],
    'val_ssim': [float(x) for x in history['val_ssim']],
    'epochs': [int(x) for x in history['epochs']]
}
history_path = os.path.join(output_dir, "metrics", "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history_json, f, indent=2)
print(f"📊 History saved to: {history_path}")

# Plot curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history['epochs'], history['train_g_loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Generator Loss')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history['epochs'], history['val_psnr'], 'g-')
plt.xlabel('Epoch')
plt.ylabel('PSNR (dB)')
plt.title('Validation PSNR')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(history['epochs'], history['val_ssim'], 'b-')
plt.xlabel('Epoch')
plt.ylabel('SSIM')
plt.title('Validation SSIM')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "metrics", "training_curves.png"), dpi=150)
plt.show()
print("📈 Training curves saved!")

In [ ]:
# 🚀 QUICK START: Load Pre-Trained Model (Skip Training)
# Run this cell instead of cell 3 if you already have a trained model

import os
import torch
import torch.nn as nn
from torchvision import models
import torch.nn.functional as F

# Architecture definitions (needed to load the model)
class ResidualBlock(nn.Module):
    def __init__(self, features, dilation=1):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(dilation),
            nn.Conv2d(features, features, 3, padding=0, dilation=dilation),
            nn.InstanceNorm2d(features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(features, features, 3, padding=0, dilation=1),
            nn.InstanceNorm2d(features)
        )
    def forward(self, x):
        return x + self.block(x)

class ModernGenerator(nn.Module):
    def __init__(self, channels=3):
        super(ModernGenerator, self).__init__()
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, 64, 7), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1), nn.InstanceNorm2d(256), nn.ReLU(True)
        ]
        model += [ResidualBlock(256, dilation=2)]
        model += [ResidualBlock(256, dilation=4)]
        model += [ResidualBlock(256, dilation=8)]
        model += [ResidualBlock(256, dilation=1)]
        model += [
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.InstanceNorm2d(64), nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, channels, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*model)
    def forward(self, x):
        return self.model(x)

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device: {device}")

generator = ModernGenerator().to(device)

# Try loading best model first, then final model
output_dir = os.path.join(os.getcwd(), "output")
best_model_path = os.path.join(output_dir, "saved_models", "generator_best.pth")
final_model_path = os.path.join(output_dir, "saved_models", "generator_modern_final.pth")

if os.path.exists(best_model_path):
    generator.load_state_dict(torch.load(best_model_path, map_location=device))
    print(f"✅ Loaded BEST model from: {best_model_path}")
elif os.path.exists(final_model_path):
    generator.load_state_dict(torch.load(final_model_path, map_location=device))
    print(f"✅ Loaded FINAL model from: {final_model_path}")
else:
    print("❌ No trained model found! Please run training cell first.")
    
generator.eval()
print("🎯 Model ready for inference!")

# 📊 Complete Satellite Image Inpainting Project

## What's Included:
- ✅ **256x256 resolution** for better quality
- ✅ **Train/Validation split** (90/10)
- ✅ **Automatic checkpointing** every 10 epochs
- ✅ **Best model saving** based on validation PSNR
- ✅ **Training history tracking** with JSON export
- ✅ **Loss curves visualization**
- ✅ **VGG Perceptual Loss** for sharp details
- ✅ **PatchGAN Discriminator** for realistic textures
- ✅ **LPIPS + FID metrics** for quality assessment

## Training Configuration:
- Image Size: 256x256
- Batch Size: 16
- Epochs: 100
- Learning Rate: 0.0002
- Validation: 10% of dataset

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2
import requests
from io import BytesIO

# --- 1. CONFIGURATION ET CHARGEMENT DU MODÈLE ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

# On recrée l'instance du modèle
model = ModernGenerator().to(device)

# Chargement des poids entraînés (updated for local path)
output_dir = os.path.join(os.path.dirname(__file__) if '__file__' in globals() else os.getcwd(), "output")
checkpoint_path = os.path.join(output_dir, "saved_models", "generator_modern_final.pth")

if os.path.exists(checkpoint_path):
    # Load with map_location for CPU compatibility if needed
    state_dict = torch.load(checkpoint_path, map_location=device)
    
    # Handle DataParallel state dict if model was trained with it
    if list(state_dict.keys())[0].startswith('module.'):
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:]  # remove 'module.' prefix
            new_state_dict[name] = v
        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(state_dict)
    
    print(f"✅ Model loaded successfully from: {checkpoint_path}")
else:
    print(f"❌ Error: Model file not found at: {checkpoint_path}")
    print("⚠️ Please train the model first by running the training cell!")

model.eval() # Mode évaluation (désactive le dropout, etc.)

# --- 2. FONCTIONS DE TRAITEMENT ---

def load_image_from_url(url):
    """Télécharge une image depuis une URL"""
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        return img
    except Exception as e:
        print(f"❌ Download error: {e}")
        return None

def predict_external_image(image_path_or_url, is_url=True):
    """
    1. Charge l'image
    2. La redimensionne à 128x128
    3. Ajoute des nuages artificiels
    4. Fait la prédiction
    5. Affiche le résultat
    """
    
    # 1. Chargement
    if is_url:
        print(f"📥 Downloading image from URL...")
        img_pil = load_image_from_url(image_path_or_url)
    else:
        print(f"📂 Loading local image: {os.path.basename(image_path_or_url)}")
        img_pil = Image.open(image_path_or_url).convert('RGB')
        
    if img_pil is None: return

    # 2. Prétraitement (Resize + Normalize)
    transform = transforms.Compose([
        transforms.Resize((128, 128), Image.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    img_tensor = transform(img_pil).unsqueeze(0).to(device) # Ajout dimension batch (1, 3, 128, 128)
    
    # 3. Création du masque (Nuages)
    masked_tensor = img_tensor.clone()
    _, _, h, w = masked_tensor.shape
    mask = np.zeros((h, w), np.float32)
    
    # Dessin des nuages artificiels
    num_strokes = 6 # Nombre de nuages
    for _ in range(num_strokes):
        x1, x2 = np.random.randint(0, w), np.random.randint(0, w)
        y1, y2 = np.random.randint(0, h), np.random.randint(0, h)
        thickness = np.random.randint(5, 15)
        cv2.line(mask, (x1, y1), (x2, y2), (1), thickness)
        cv2.circle(mask, (np.random.randint(0, w), np.random.randint(0, h)), np.random.randint(5, 15), (1), -1)

    mask_torch = torch.from_numpy(mask).to(device)
    masked_tensor[0, 0, mask_torch == 1] = 1.0 # Canal R
    masked_tensor[0, 1, mask_torch == 1] = 1.0 # Canal G
    masked_tensor[0, 2, mask_torch == 1] = 1.0 # Canal B

    # 4. Prédiction (Inpainting)
    print("🧠 Running inference...")
    with torch.no_grad():
        gen_tensor = model(masked_tensor)

    # 5. Affichage
    def to_numpy(tensor):
        img = tensor.squeeze(0).cpu().detach().numpy()
        img = np.transpose(img, (1, 2, 0)) # CHW -> HWC
        img = (img + 1) / 2.0
        return np.clip(img, 0, 1)

    original = to_numpy(img_tensor)
    masked = to_numpy(masked_tensor)
    reconstructed = to_numpy(gen_tensor)

    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(original)
    plt.title("Image Originale", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.imshow(masked)
    plt.title("Avec Nuages (Input)", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.imshow(reconstructed)
    plt.title("Reconstruction IA", fontsize=12, fontweight='bold')
    plt.axis('off')

    plt.tight_layout()
    plt.show()
    print("✅ Results displayed!")

# --- 3. EXEMPLE D'UTILISATION ---

# Option A : Tester avec une image satellite depuis Internet
url_test = "https://images.unsplash.com/photo-1477959858617-67f85cf4f1df?q=80&w=1000&auto=format&fit=crop"
print("🌐 Testing with online image:")
predict_external_image(url_test, is_url=True)

# Option B : Test local (uncomment to use)
# local_test = r"C:\Users\MAROUAN\Pictures\my_satellite_image.jpg"
# if os.path.exists(local_test):
#     predict_external_image(local_test, is_url=False)

In [ ]:
# Test with local dataset image

# Update this to match your dataset location
dataset_base = r"C:\Users\MAROUAN\Downloads\Project\UCMerced_LandUse"
local_path = os.path.join(dataset_base, "Images", "buildings", "buildings01.tif")

# Try alternative extensions if .tif doesn't exist
if not os.path.exists(local_path):
    for ext in ['.jpg', '.png', '.jpeg']:
        alt_path = local_path.replace('.tif', ext)
        if os.path.exists(alt_path):
            local_path = alt_path
            break

if os.path.exists(local_path):
    print(f"🖼️ Testing with local image: {os.path.basename(local_path)}")
    predict_external_image(local_path, is_url=False)
else:
    print(f"⚠️ File not found: {local_path}")
    print("\n🔍 Searching for any available image in dataset...")
    
    # Try to find any image in the dataset
    if os.path.exists(dataset_base):
        for root, dirs, files in os.walk(dataset_base):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.tif', '.jpeg')):
                    found_path = os.path.join(root, file)
                    print(f"✅ Found: {found_path}")
                    predict_external_image(found_path, is_url=False)
                    break
            else:
                continue
            break
    else:
        print(f"❌ Dataset directory not found: {dataset_base}")
        print("Please update the dataset_base path in this cell.")